In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

In [4]:
BASE_DIR=Path("..")/'data'
train_transaction = pd.read_csv(BASE_DIR/'train_transaction.csv')
train_identity = pd.read_csv(BASE_DIR / 'train_identity.csv')

print(f"Transaction data shape: {train_transaction.shape}")
print(f"Identity data shape: {train_identity.shape}")

Transaction data shape: (590540, 394)
Identity data shape: (144233, 41)


### Transaction

In [6]:
train_transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 1.7+ GB


In [ ]:
#  Drop V-features
v_cols = [col for col in train_transaction.columns if col.startswith('V')]
df_transaction = train_transaction.drop(columns=v_cols).copy()

# Engineer Time Features (Days and Hours)
df_transaction['Transaction_Day'] = np.floor((df_transaction['TransactionDT'] / (3600 * 24)) - 1)
df_transaction['Transaction_Hour'] = np.floor(df_transaction['TransactionDT'] / 3600) % 24
df_transaction = df_transaction.drop('TransactionDT', axis=1)

# Define Categorical vs. Numerical
trans_cat_cols = ['ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 
                  'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain'] + [f"M{i}" for i in range(1, 10)]
trans_cat_cols = [col for col in trans_cat_cols if col in df_transaction.columns] # Ensure columns actually exist in the dataframe to prevent KeyError
trans_num_cols = [col for col in df_transaction.columns if col not in trans_cat_cols and col not in ['TransactionID', 'isFraud']]

# Impute
print("Imputing transaction NAs...")
cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
df_transaction[trans_cat_cols] = cat_imputer.fit_transform(df_transaction[trans_cat_cols])

num_imputer = SimpleImputer(strategy='median')
df_transaction[trans_num_cols] = num_imputer.fit_transform(df_transaction[trans_num_cols])

print(f"Remaining NAs in transactions: {df_transaction.isna().sum().max()}") # Should be 0

Imputing transaction NAs...
Remaining NAs in transactions: 0


In [14]:
df_transaction.info()
df_transaction.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 56 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   TransactionID     590540 non-null  int64  
 1   isFraud           590540 non-null  int64  
 2   TransactionAmt    590540 non-null  float64
 3   ProductCD         590540 non-null  object 
 4   card1             590540 non-null  object 
 5   card2             590540 non-null  object 
 6   card3             590540 non-null  object 
 7   card4             590540 non-null  object 
 8   card5             590540 non-null  object 
 9   card6             590540 non-null  object 
 10  addr1             590540 non-null  object 
 11  addr2             590540 non-null  object 
 12  dist1             590540 non-null  float64
 13  dist2             590540 non-null  float64
 14  P_emaildomain     590540 non-null  object 
 15  R_emaildomain     590540 non-null  object 
 16  C1                59

,TransactionID,isFraud,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,M2,M3,M4,M5,M6,M7,M8,M9,Transaction_Day,Transaction_Hour
0,2987000,0,68.5,W,13926,missing,150.0,discover,142.0,credit,...,T,T,M2,F,T,missing,missing,missing,0.0,0.0
1,2987001,0,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,...,missing,missing,M0,T,T,missing,missing,missing,0.0,0.0
2,2987002,0,59.0,W,4663,490.0,150.0,visa,166.0,debit,...,T,T,M0,F,F,F,F,F,0.0,0.0
3,2987003,0,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,...,missing,missing,M0,T,F,missing,missing,missing,0.0,0.0
4,2987004,0,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,...,missing,missing,missing,missing,missing,missing,missing,missing,0.0,0.0
5,2987005,0,49.0,W,5937,555.0,150.0,visa,226.0,debit,...,T,T,M1,F,T,missing,missing,missing,0.0,0.0
6,2987006,0,159.0,W,12308,360.0,150.0,visa,166.0,debit,...,T,T,M0,F,F,T,T,T,0.0,0.0
7,2987007,0,422.5,W,12695,490.0,150.0,visa,226.0,debit,...,missing,missing,M0,F,F,missing,missing,missing,0.0,0.0
8,2987008,0,15.0,H,2803,100.0,150.0,visa,226.0,debit,...,missing,missing,missing,missing,missing,missing,missing,missing,0.0,0.0
9,2987009,0,117.0,W,17399,111.0,150.0,mastercard,224.0,debit,...,T,T,M0,T,T,missing,missing,missing,0.0,0.0


### Identity

In [ ]:
df_identity = train_identity.copy()

# 1. Define Categorical vs. Numerical for Identity
id_cat_cols = ['DeviceType', 'DeviceInfo'] + [f"id_{i}" for i in range(12, 39)]
id_cat_cols = [col for col in id_cat_cols if col in df_identity.columns]

id_num_cols = [col for col in df_identity.columns if col not in id_cat_cols and col != 'TransactionID']

# 2. Impute
print("Imputing identity NAs...")
df_identity[id_cat_cols] = cat_imputer.fit_transform(df_identity[id_cat_cols])
df_identity[id_num_cols] = num_imputer.fit_transform(df_identity[id_num_cols])

print(f"Remaining NAs in identity: {df_identity.isna().sum().max()}") # Should be 0

Imputing identity NAs...
Remaining NAs in identity: 0


In [16]:
df_identity.info()
df_identity.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144233 entries, 0 to 144232
Data columns (total 41 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionID  144233 non-null  int64  
 1   id_01          144233 non-null  float64
 2   id_02          144233 non-null  float64
 3   id_03          144233 non-null  float64
 4   id_04          144233 non-null  float64
 5   id_05          144233 non-null  float64
 6   id_06          144233 non-null  float64
 7   id_07          144233 non-null  float64
 8   id_08          144233 non-null  float64
 9   id_09          144233 non-null  float64
 10  id_10          144233 non-null  float64
 11  id_11          144233 non-null  float64
 12  id_12          144233 non-null  object 
 13  id_13          144233 non-null  object 
 14  id_14          144233 non-null  object 
 15  id_15          144233 non-null  object 
 16  id_16          144233 non-null  object 
 17  id_17          144233 non-nul

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,0.0,0.0,0.0,0.0,14.0,-34.0,0.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,0.0,0.0,0.0,-5.0,14.0,-34.0,0.0,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,14.0,-34.0,0.0,...,chrome 62.0,missing,missing,missing,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,0.0,0.0,0.0,-6.0,14.0,-34.0,0.0,...,chrome 62.0,missing,missing,missing,F,F,T,T,desktop,missing
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,14.0,-34.0,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS
5,2987017,-5.0,61141.0,3.0,0.0,3.0,0.0,14.0,-34.0,3.0,...,chrome 62.0,24.0,1366x768,match_status:2,T,F,T,T,desktop,Windows
6,2987022,-15.0,125800.5,0.0,0.0,0.0,0.0,14.0,-34.0,0.0,...,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing
7,2987038,0.0,31964.0,0.0,0.0,0.0,-10.0,14.0,-34.0,0.0,...,chrome 62.0,32.0,1920x1080,match_status:2,T,F,T,T,mobile,missing
8,2987040,-10.0,116098.0,0.0,0.0,0.0,0.0,14.0,-34.0,0.0,...,chrome 62.0,missing,missing,missing,F,F,T,T,desktop,Windows
9,2987048,-5.0,257037.0,0.0,0.0,0.0,0.0,14.0,-34.0,0.0,...,chrome 62.0,missing,missing,missing,F,F,T,T,desktop,Windows
